# 🚚 Оптимизация маршрутов доставки
## Алгоритм «Ближайшего соседа» (Nearest Neighbor Heuristic)

**Датасет:** Brazilian E-Commerce Public Dataset by Olist — Kaggle  
**Ссылка:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

---
### Цели:
1. Загрузить реальные данные доставок с Kaggle
2. Подготовить координаты точек доставки
3. Реализовать алгоритм Ближайшего соседа
4. Сравнить с случайным маршрутом и NN+2-opt
5. Экспортировать модель и данные для веб-приложения с Яндекс Картами

## 1. Установка зависимостей

In [ ]:
!pip install kaggle pandas numpy matplotlib seaborn folium scikit-learn scipy joblib tqdm

## 2. Загрузка датасета с Kaggle

> **Инструкция по настройке Kaggle API:**
> 1. Откройте https://www.kaggle.com → Account → API → **Create New Token**
> 2. Скачается файл `kaggle.json`
> 3. Поместите его по пути:
>    - Linux/Mac: `~/.kaggle/kaggle.json`
>    - Windows: `C:\Users\<user>\.kaggle\kaggle.json`
> 4. Запустите ячейку ниже

In [ ]:
import os

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
os.makedirs('./data', exist_ok=True)

# Скачивание датасета Olist (Brazilian E-Commerce)
!kaggle datasets download -d olistbr/brazilian-ecommerce -p ./data --unzip

print('✅ Датасет загружен в папку ./data/')
print('Файлы:', os.listdir('./data/'))

## 3. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import json
import joblib
import time
import math
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (13, 7)
plt.rcParams['font.size'] = 12

print('✅ Все библиотеки импортированы')

## 4. Загрузка и первичный анализ данных (EDA)

In [ ]:
DATA_PATH = './data/'

customers   = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
geolocation = pd.read_csv(DATA_PATH + 'olist_geolocation_dataset.csv')
orders      = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
sellers     = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')

print('Размеры датасетов:')
print(f'  customers:   {customers.shape}')
print(f'  geolocation: {geolocation.shape}')
print(f'  orders:      {orders.shape}')
print(f'  sellers:     {sellers.shape}')

display(geolocation.head())

In [ ]:
print('Статистика координат:')
display(geolocation[['geolocation_lat','geolocation_lng']].describe())
print(f'\nУникальных городов: {geolocation["geolocation_city"].nunique()}')
print(f'Уникальных штатов:  {geolocation["geolocation_state"].nunique()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_cities = geolocation['geolocation_city'].value_counts().head(15)
axes[0].barh(top_cities.index[::-1], top_cities.values[::-1],
             color=sns.color_palette('rocket', 15))
axes[0].set_title('Топ-15 городов по количеству точек', fontweight='bold')
axes[0].set_xlabel('Количество точек доставки')

state_counts = geolocation['geolocation_state'].value_counts()
axes[1].pie(state_counts.values[:10], labels=state_counts.index[:10],
            autopct='%1.1f%%', startangle=90,
            colors=sns.color_palette('husl', 10))
axes[1].set_title('Доля точек по штатам (топ-10)', fontweight='bold')

plt.tight_layout()
plt.savefig('./data/eda_cities.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Тепловая карта плотности точек доставки по Бразилии
fig, ax = plt.subplots(figsize=(12, 10))

geo_clean = geolocation[
    (geolocation['geolocation_lat'].between(-35, 5)) &
    (geolocation['geolocation_lng'].between(-75, -30))
].sample(50000, random_state=42)

ax.scatter(geo_clean['geolocation_lng'], geo_clean['geolocation_lat'],
           alpha=0.05, s=1, c='#FF6B35')
ax.set_facecolor('#0D1117')
fig.patch.set_facecolor('#0D1117')
ax.set_title('Карта плотности точек доставки — Бразилия',
             color='white', fontsize=14, fontweight='bold')
ax.tick_params(colors='white')
ax.set_xlabel('Долгота', color='white')
ax.set_ylabel('Широта', color='white')

plt.tight_layout()
plt.savefig('./data/eda_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()

## 5. Подготовка точек доставки (Сан-Паулу)

In [ ]:
# Фильтрация по Сан-Паулу + очистка выбросов
sp_geo = geolocation[
    geolocation['geolocation_city'].str.lower().str.contains('sao paulo')
].copy()

sp_geo = sp_geo[
    (sp_geo['geolocation_lat'].between(-24.1, -23.3)) &
    (sp_geo['geolocation_lng'].between(-46.9, -46.3))
]

# Усредняем координаты по индексу
sp_geo = sp_geo.groupby('geolocation_zip_code_prefix').agg(
    lat  = ('geolocation_lat', 'mean'),
    lng  = ('geolocation_lng', 'mean'),
    city = ('geolocation_city', 'first')
).reset_index()

print(f'Уникальных точек в Сан-Паулу: {len(sp_geo)}')
display(sp_geo.head())

In [ ]:
NUM_POINTS = 30   # Измените: 20–200 точек

np.random.seed(42)
sample = sp_geo.sample(NUM_POINTS).reset_index(drop=True)
sample['id']      = range(len(sample))
sample['label']   = ['🏭 Склад' if i == 0 else f'Точка {i}' for i in range(len(sample))]
sample['address'] = sample['geolocation_zip_code_prefix'].astype(str) + ', São Paulo, Brazil'

print(f'Выборка: {NUM_POINTS} точек')
print(f'Склад (депо): lat={sample.iloc[0]["lat"]:.4f}, lng={sample.iloc[0]["lng"]:.4f}')
display(sample[['id','label','lat','lng','address']].head(5))

## 6. Реализация алгоритмов

In [ ]:
# ══════════════════════════════════════════════════════════════
# Вспомогательные функции
# ══════════════════════════════════════════════════════════════

def haversine(lat1, lng1, lat2, lng2):
    """Расстояние между двумя точками на сфере Земли (км)."""
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lng2 - lng1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return 2 * R * math.asin(math.sqrt(a))

def build_distance_matrix(df):
    """Матрица попарных расстояний N×N."""
    n = len(df)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(i+1, n):
            d = haversine(df.iloc[i]['lat'], df.iloc[i]['lng'],
                          df.iloc[j]['lat'], df.iloc[j]['lng'])
            D[i, j] = D[j, i] = d
    return D

def route_distance(route, D):
    """Суммарная длина замкнутого маршрута."""
    total = sum(D[route[i], route[i+1]] for i in range(len(route)-1))
    total += D[route[-1], route[0]]  # возврат в начало
    return total

print('Строю матрицу расстояний...')
D = build_distance_matrix(sample)
np.save('./data/distance_matrix.npy', D)
print(f'✅ Матрица {D.shape[0]}×{D.shape[0]} готова. D[0,1]={D[0,1]:.2f} км')

In [ ]:
# ══════════════════════════════════════════════════════════════
# АЛГОРИТМ 1: Ближайший сосед (Nearest Neighbor)
#
# Принцип: из текущей точки идём в ближайшую непосещённую.
# Сложность: O(n²). Запускаем от каждой стартовой точки,
# сохраняем лучший результат.
# ══════════════════════════════════════════════════════════════

class NearestNeighborRouter:
    def __init__(self):
        self.route_      = None
        self.total_dist_ = None
        self.exec_time_  = None
        self.history_    = []

    def _nn_from(self, D, start):
        n = D.shape[0]
        visited = [False] * n
        route = [start]
        visited[start] = True
        for _ in range(n - 1):
            cur = route[-1]
            best_j, best_d = -1, float('inf')
            for j in range(n):
                if not visited[j] and D[cur, j] < best_d:
                    best_d, best_j = D[cur, j], j
            route.append(best_j)
            visited[best_j] = True
        return route

    def fit(self, D, try_all_starts=True):
        t0 = time.time()
        n  = D.shape[0]
        starts = range(n) if try_all_starts else [0]
        best_route, best_dist = None, float('inf')
        for s in tqdm(starts, desc='NN — перебор стартов'):
            r = self._nn_from(D, s)
            d = route_distance(r, D)
            self.history_.append({'start': s, 'distance': d})
            if d < best_dist:
                best_dist, best_route = d, r
        self.route_      = best_route
        self.total_dist_ = best_dist
        self.exec_time_  = time.time() - t0
        return self

    def predict(self, D, start=0):
        """Применить алгоритм к новой матрице расстояний."""
        return self._nn_from(D, start)


nn_model = NearestNeighborRouter()
nn_model.fit(D, try_all_starts=True)

print(f'\n✅ Nearest Neighbor:')
print(f'   Расстояние:   {nn_model.total_dist_:.2f} км')
print(f'   Время:        {nn_model.exec_time_:.3f} сек')

In [ ]:
# ══════════════════════════════════════════════════════════════
# АЛГОРИТМ 2: Случайный маршрут (Baseline)
# ══════════════════════════════════════════════════════════════

def random_route(D, n_trials=300, seed=42):
    np.random.seed(seed)
    n = D.shape[0]
    best_route, best_dist = None, float('inf')
    for _ in range(n_trials):
        r = list(np.random.permutation(n))
        d = route_distance(r, D)
        if d < best_dist:
            best_dist, best_route = d, r
    return best_route, best_dist

rand_route, rand_dist = random_route(D)
print(f'✅ Случайный маршрут (лучший из 300): {rand_dist:.2f} км')

In [ ]:
# ══════════════════════════════════════════════════════════════
# АЛГОРИТМ 3: NN + 2-opt (локальное улучшение)
#
# Итеративно переворачиваем подотрезки маршрута, пока
# суммарное расстояние уменьшается.
# ══════════════════════════════════════════════════════════════

def two_opt(route, D, max_iter=2000):
    best = list(route)
    improved = True
    iters = 0
    while improved and iters < max_iter:
        improved = False
        iters += 1
        for i in range(1, len(best) - 1):
            for j in range(i + 1, len(best)):
                new_route = best[:i] + best[i:j+1][::-1] + best[j+1:]
                if route_distance(new_route, D) < route_distance(best, D):
                    best = new_route
                    improved = True
    return best

t0 = time.time()
opt2_route = two_opt(nn_model.route_, D)
opt2_dist  = route_distance(opt2_route, D)
opt2_time  = time.time() - t0

print(f'✅ NN + 2-opt:')
print(f'   Расстояние:   {opt2_dist:.2f} км')
print(f'   Время 2-opt:  {opt2_time:.3f} сек')

## 7. Сравнительный анализ

In [ ]:
results = pd.DataFrame([
    {'Алгоритм': 'Случайный маршрут',    'Расстояние (км)': rand_dist,           'Улучшение (%)': 0},
    {'Алгоритм': 'Ближайший сосед (NN)', 'Расстояние (км)': nn_model.total_dist_, 'Улучшение (%)': (rand_dist - nn_model.total_dist_)/rand_dist*100},
    {'Алгоритм': 'NN + 2-opt',           'Расстояние (км)': opt2_dist,            'Улучшение (%)': (rand_dist - opt2_dist)/rand_dist*100},
])
results = results.round(2)

display(results.style
    .background_gradient(subset=['Расстояние (км)'], cmap='RdYlGn_r')
    .background_gradient(subset=['Улучшение (%)'],   cmap='RdYlGn')
    .format({'Расстояние (км)': '{:.2f}', 'Улучшение (%)': '{:.1f}%'}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

algos  = ['Случайный\nмаршрут', 'Ближайший\nсосед (NN)', 'NN + 2-opt']
dists  = [rand_dist, nn_model.total_dist_, opt2_dist]
colors = ['#e74c3c', '#f39c12', '#2ecc71']

bars = axes[0].bar(algos, dists, color=colors, width=0.5, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, dists):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f} км', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Суммарное расстояние маршрута', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Расстояние (км)')
axes[0].set_ylim(0, max(dists) * 1.15)

hist_df = pd.DataFrame(nn_model.history_)
axes[1].plot(hist_df['start'], hist_df['distance'], color='#3498db', alpha=0.5, linewidth=1)
axes[1].axhline(nn_model.total_dist_, color='#2ecc71', linewidth=2, linestyle='--',
                label=f'Лучший: {nn_model.total_dist_:.1f} км')
axes[1].set_title('NN: длина маршрута при разных стартовых точках', fontweight='bold')
axes[1].set_xlabel('Стартовая точка')
axes[1].set_ylabel('Расстояние (км)')
axes[1].legend()

plt.tight_layout()
plt.savefig('./data/comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Визуализация маршрутов (Folium)

In [ ]:
def draw_route_map(df, route, title, color='blue', filename=None):
    center = [df['lat'].mean(), df['lng'].mean()]
    m = folium.Map(location=center, zoom_start=11, tiles='CartoDB positron')

    coords = [(df.iloc[r]['lat'], df.iloc[r]['lng']) for r in route]
    coords.append(coords[0])
    folium.PolyLine(coords, color=color, weight=3, opacity=0.85).add_to(m)

    for idx, row in df.iterrows():
        order_num  = route.index(idx) + 1
        icon_color = 'red' if idx == 0 else 'blue'
        icon_name  = 'home' if idx == 0 else 'circle'
        folium.Marker(
            [row['lat'], row['lng']],
            popup=f"#{order_num} {row['label']}",
            tooltip=f"#{order_num} {row['label']}",
            icon=folium.Icon(color=icon_color, icon=icon_name, prefix='fa')
        ).add_to(m)

    total_d = route_distance(route, D)
    folium.map.Marker(
        [center[0]+0.04, center[1]],
        icon=folium.DivIcon(
            html=f'<div style="font-size:13px;font-weight:bold;background:rgba(255,255,255,0.85);'
                 f'padding:6px 10px;border-radius:6px;border:1px solid #ccc;">'
                 f'{title} — {total_d:.2f} км</div>'
        )
    ).add_to(m)

    if filename:
        m.save(f'./data/{filename}.html')
        print(f'  Карта сохранена: ./data/{filename}.html')
    return m


draw_route_map(sample, rand_route,       'Случайный',   'red',   'map_random')
draw_route_map(sample, nn_model.route_,  'NN',          'blue',  'map_nn')
draw_route_map(sample, opt2_route,       'NN+2opt',     'green', 'map_nn2opt')
print('✅ Все карты сохранены')

## 9. Тест масштабируемости

In [ ]:
sizes    = [10, 20, 30, 50, 75, 100, 150, 200]
nn_times = []
nn_dists = []

for n in tqdm(sizes, desc='Масштабируемость'):
    sub = sp_geo.sample(n, random_state=42).reset_index(drop=True)
    Dn  = build_distance_matrix(sub)
    m   = NearestNeighborRouter()
    m.fit(Dn, try_all_starts=False)
    nn_times.append(m.exec_time_)
    nn_dists.append(m.total_dist_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(sizes, nn_times, 'o-', color='#e74c3c', linewidth=2, markersize=8)
axes[0].set_title('Время работы алгоритма NN', fontweight='bold')
axes[0].set_xlabel('Количество точек (N)')
axes[0].set_ylabel('Время (сек)')

axes[1].plot(sizes, nn_dists, 's-', color='#3498db', linewidth=2, markersize=8)
axes[1].set_title('Длина маршрута NN', fontweight='bold')
axes[1].set_xlabel('Количество точек (N)')
axes[1].set_ylabel('Расстояние (км)')

plt.tight_layout()
plt.savefig('./data/scalability.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Экспорт модели и данных для сайта с Яндекс Картами

In [ ]:
# Сохраняем модель
joblib.dump(nn_model, './data/nn_model.pkl')
print('✅ Модель сохранена: ./data/nn_model.pkl')

# JSON-экспорт для веб-приложения
# ВАЖНО: Для отображения на Яндекс Картах точки должны быть
# в реальных координатах. Ниже мы используем координаты
# Сан-Паулу из датасета — для российского города замените
# на нужные координаты или загрузите другой датасет.

export = {
    'meta': {
        'dataset':   'Olist Brazilian E-Commerce (Kaggle)',
        'city':      'São Paulo',
        'n_points':  NUM_POINTS,
        'generated': str(pd.Timestamp.now())
    },
    'points': [
        {
            'id':      int(row['id']),
            'label':   row['label'],
            'address': row['address'],
            'lat':     float(row['lat']),
            'lng':     float(row['lng'])
        }
        for _, row in sample.iterrows()
    ],
    'routes': {
        'random': {
            'order':    rand_route,
            'distance': round(rand_dist, 2)
        },
        'nearest_neighbor': {
            'order':    nn_model.route_,
            'distance': round(nn_model.total_dist_, 2)
        },
        'nn_2opt': {
            'order':    opt2_route,
            'distance': round(opt2_dist, 2)
        }
    },
    'stats': {
        'random_dist':       round(rand_dist, 2),
        'nn_dist':           round(nn_model.total_dist_, 2),
        'nn2opt_dist':       round(opt2_dist, 2),
        'nn_improvement':    round((rand_dist - nn_model.total_dist_)/rand_dist*100, 1),
        'nn2opt_improvement':round((rand_dist - opt2_dist)/rand_dist*100, 1),
        'exec_time_sec':     round(nn_model.exec_time_, 4)
    }
}

with open('./data/route_data.json', 'w', encoding='utf-8') as f:
    json.dump(export, f, ensure_ascii=False, indent=2)

print('✅ route_data.json сохранён для сайта с Яндекс Картами')
print()
print('📊 Итоги:')
print(f'   Случайный маршрут:  {rand_dist:.1f} км')
print(f'   Ближайший сосед:    {nn_model.total_dist_:.1f} км  (−{export["stats"]["nn_improvement"]}%)')
print(f'   NN + 2-opt:         {opt2_dist:.1f} км  (−{export["stats"]["nn2opt_improvement"]}%)')
print()
print('🌐 Откройте index.html — данные подгрузятся автоматически.')

In [ ]:
# ── Проверка загрузки сохранённой модели ──────────────────────
loaded_model = joblib.load('./data/nn_model.pkl')

test_sample = sp_geo.sample(10, random_state=99).reset_index(drop=True)
D_test      = build_distance_matrix(test_sample)
test_route  = loaded_model.predict(D_test, start=0)
test_dist   = route_distance(test_route, D_test)

print('✅ Модель загружена и применена к новым данным')
print(f'   Маршрут:     {test_route}')
print(f'   Расстояние:  {test_dist:.2f} км')
print()
print('─' * 55)
print('🎓 Notebook завершён! Файлы готовы:')
print('   ./data/nn_model.pkl      — обученная модель')
print('   ./data/route_data.json   — данные для сайта')
print('   ./data/distance_matrix.npy')
print('   ./data/map_nn.html, map_random.html, map_nn2opt.html')